# MDGD Encoder Baseline: Fine-Tuned SciBERT / DeBERTa-v3

Fine-tunes an encoder (SciBERT or DeBERTa-v3-base) on binary edge classification for the MDGD derivation-graph dataset.
5-fold article-level cross-validation × 3 seeds, pooled precision/recall/F1 matching the paper's protocol.

**Expected runtime on Colab free T4:**
- SciBERT, 5-fold × 3 seeds: ~4–6 h  |  DeBERTa-v3-base: ~5–8 h
- `DEBUG_MODE=True` (1 fold, 1 epoch): ~10–15 min

**Required files:**
- `mdgd.json` — adjacency-list dataset (64 articles)
- `articles/{article_id}.html` — downloaded automatically from ar5iv if missing


## 1. Setup

In [ ]:
%%capture
!pip install transformers accelerate torch scikit-learn beautifulsoup4 lxml requests tqdm


In [ ]:
# Uncomment to mount Google Drive if your files live there
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.chdir('/content/drive/MyDrive/mdgd_data')


## 2. Configuration

In [ ]:
import json, os, re, warnings, random, logging, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import requests
import torch
import torch.nn as nn
from bs4 import BeautifulSoup
from sklearn.model_selection import KFold
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Device: {DEVICE}')
if torch.cuda.is_available():
    logger.info(f'GPU: {torch.cuda.get_device_name(0)}')


@dataclass
class Config:
    # ── Model ──
    MODEL_NAME: str = 'allenai/scibert_scivocab_uncased'
    # MODEL_NAME: str = 'microsoft/deberta-v3-base'
    MAX_LENGTH: int = 512

    # ── Training ──
    BATCH_SIZE: int = 16          # reduce to 8 if OOM
    EPOCHS: int = 5
    ENCODER_LR: float = 2e-5
    HEAD_LR: float = 1e-4
    POSITIVE_WEIGHT: float = 15.0
    PATIENCE: int = 2
    WARMUP_RATIO: float = 0.1
    DROPOUT: float = 0.1
    GRAD_CLIP: float = 1.0

    # ── CV & seeds ──
    N_FOLDS: int = 5
    SEEDS: List[int] = field(default_factory=lambda: [42, 123, 456])
    THRESHOLD: float = 0.5

    # ── Paths ──
    DATASET_PATH: str = 'mdgd.json'
    ARTICLES_DIR: str = 'articles'
    RESULTS_PATH: str = 'encoder_baseline_results.json'
    CHECKPOINT_DIR: str = 'checkpoints'

    # ── Debug ──
    DEBUG_MODE: bool = True   # True = 1 fold, 1 epoch to validate pipeline end-to-end


cfg = Config()
Path(cfg.CHECKPOINT_DIR).mkdir(exist_ok=True)
Path(cfg.ARTICLES_DIR).mkdir(exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


logger.info(f'Config: model={cfg.MODEL_NAME}, debug={cfg.DEBUG_MODE}')


## 3. Data Loading

In [ ]:
_PATTERN      = re.compile(r'((?:Sx?\d+|A\d+))\.(E\d+)')
_PATTERN_BARE = re.compile(r'^(E\d+)\.')


def load_dataset(path: str) -> Dict[str, Any]:
    """Load mdgd.json -> dict keyed by Article ID."""
    with open(path, 'r') as f:
        articles = json.load(f)
    return {a['Article ID']: a for a in articles}


def fetch_article_html(article_id: str, articles_dir: str) -> Optional[str]:
    """Return HTML from local cache or download from ar5iv."""
    safe_id = article_id.replace('/', '_')
    local = Path(articles_dir) / f'{safe_id}.html'
    if local.exists():
        return local.read_text(encoding='utf-8', errors='replace')
    url = f'https://ar5iv.org/abs/{article_id}'
    logger.warning(f'Downloading {article_id} from ar5iv...')
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        local.write_text(resp.text, encoding='utf-8')
        return resp.text
    except Exception as e:
        logger.error(f'Failed to download {article_id}: {e}')
        return None


def _eq_key(raw_id: str) -> Optional[str]:
    """'S3.E1.m1' -> 'S3.E1'; bare 'E1.foo' -> 'S0.E1'."""
    m = _PATTERN.search(raw_id)
    if m:
        return f'{m.group(1)}.{m.group(2)}'
    m = _PATTERN_BARE.match(raw_id)
    if m:
        return f'S0.{m.group(1)}'
    return None


def parse_article_html(html: str):
    """
    Parse ar5iv HTML. Returns:
      equations       : {eq_key: {'alttext': str}}
      equation_indexing: ordered list of eq_keys
      words_between   : words_between[k] = text after equation k-1
                        words_between[0] = text before first equation
      eq_paragraph    : {eq_key: id(nearest block ancestor)} for same-para check
    """
    soup = BeautifulSoup(html, 'lxml')
    equations: Dict[str, Any] = {}
    eq_indexing: List[str] = []
    words_between: List[str] = []
    eq_paragraph: Dict[str, int] = {}
    last_eq_id = 'none'
    last_update_id = 'none'

    for item in soup.recursiveChildGenerator():
        if item.name == 'math':
            raw_id  = item.get('id', '')
            alttext = item.get('alttext', '')
            key = _eq_key(raw_id)
            if key is None:
                continue
            last_eq_id = raw_id
            if key not in equations:
                equations[key] = {'alttext': '', '_parts': []}
                eq_indexing.append(key)
                para_id = None
                for parent in item.parents:
                    if parent.name in ('p', 'div', 'section', 'article', 'td'):
                        para_id = id(parent)
                        break
                eq_paragraph[key] = para_id
            equations[key]['_parts'].append(alttext)
            equations[key]['alttext'] = ' '.join(equations[key]['_parts'])

        elif isinstance(item, str):
            text = str(item)
            if last_eq_id == 'none':
                if words_between:
                    words_between[-1] += text
                else:
                    words_between.append(text)
            else:
                if last_eq_id != last_update_id:
                    words_between.append(text)
                else:
                    words_between[-1] += text
            last_update_id = last_eq_id

    return equations, eq_indexing, words_between, eq_paragraph


def load_all_articles(dataset: Dict, cfg: Config):
    """
    Load and validate all articles.
    Skips articles where any annotated equation ID is missing from HTML
    (matches the paper's protocol of evaluating on 63 articles).
    """
    articles_used = []
    article_data  = []

    for article_id, article in dataset.items():
        html = fetch_article_html(article_id, cfg.ARTICLES_DIR)
        if html is None:
            logger.warning(f'SKIP {article_id}: no HTML, performance will be degraded')
            continue

        equations, eq_indexing, words_between, eq_paragraph = parse_article_html(html)

        annotated = set(article['Equation ID'])
        missing   = annotated - set(eq_indexing)
        if missing:
            logger.warning(f'SKIP {article_id}: {len(missing)} IDs not in HTML: {missing}')
            continue

        filtered = [e for e in eq_indexing if e in annotated]
        articles_used.append(article_id)
        article_data.append({
            'article_id':       article_id,
            'equations':        equations,
            'equation_indexing': filtered,
            'words_between':    words_between,
            'eq_paragraph':     eq_paragraph,
            'adjacency_list':   article['Adjacency List'],
        })

    logger.info(f'Loaded {len(articles_used)}/{len(dataset)} articles')
    return articles_used, article_data


dataset       = load_dataset(cfg.DATASET_PATH)
articles_used, article_data = load_all_articles(dataset, cfg)
logger.info(f'Articles ready: {len(articles_used)}')


## 4. Feature Construction

In [ ]:
def _split_sentences(text: str) -> List[str]:
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]


def build_context_between(
    i: int, j: int,
    words_between: List[str],
    n_ctx: int = 2,
) -> str:
    """
    Last n_ctx sentences before eq_i
    + all text between eq_i and eq_j
    + first n_ctx sentences after eq_j.
    words_between[k] = text that appears after equation k-1 (before equation k).
    """
    parts = []
    if i < len(words_between):
        sents = _split_sentences(words_between[i])
        parts.append(' '.join(sents[-n_ctx:]))
    mid = []
    for k in range(i + 1, j + 1):
        if k < len(words_between):
            mid.append(words_between[k])
    parts.append(' '.join(mid))
    if j + 1 < len(words_between):
        sents = _split_sentences(words_between[j + 1])
        parts.append(' '.join(sents[:n_ctx]))
    return ' '.join(p for p in parts if p).strip()


def build_pairs_for_article(data: Dict, cfg: Config) -> List[Dict]:
    """All ordered pairs (vi, vj) with i < j for one article, with labels and features."""
    eq_idx      = data['equation_indexing']
    equations   = data['equations']
    wb          = data['words_between']
    eq_para     = data['eq_paragraph']
    adj         = data['adjacency_list']
    n           = len(eq_idx)
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            vi, vj = eq_idx[i], eq_idx[j]
            eq_i = equations[vi]['alttext'] if vi in equations else vi
            eq_j = equations[vj]['alttext'] if vj in equations else vj
            ctx  = build_context_between(i, j, wb)
            dist_norm = (j - i) / max(n - 1, 1)
            same_para = float(
                eq_para.get(vi) is not None
                and eq_para.get(vi) == eq_para.get(vj)
            )
            label = int(vj in (adj.get(vi) or []))
            pairs.append({
                'article_id':      data['article_id'],
                'vi': vi, 'vj': vj,
                'eq_i_text':       eq_i,
                'eq_j_text':       eq_j,
                'context':         ctx,
                'scalar_features': [dist_norm, same_para],
                'label':           label,
            })
    return pairs


all_pairs_per_article = [build_pairs_for_article(d, cfg) for d in article_data]
total_pairs = sum(len(p) for p in all_pairs_per_article)
total_pos   = sum(sum(p['label'] for p in ps) for ps in all_pairs_per_article)
logger.info(f'Pairs: {total_pairs}, positive: {total_pos} ({100*total_pos/total_pairs:.1f}%)')


## 5. Dataset Class

In [ ]:
class PairDataset(Dataset):
    """
    Tokenizes equation pairs with priority truncation:
    both equations are kept intact; context is truncated from the middle.
    Input format: [CLS] eq_i [SEP] context [SEP] eq_j [SEP]
    """

    def __init__(self, pairs: List[Dict], tokenizer, max_length: int = 512):
        self.pairs      = pairs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.pairs)

    def _fit_context(self, eq_i: str, ctx: str, eq_j: str) -> str:
        """Truncate ctx from the middle to fit within token budget."""
        tok = self.tokenizer
        n_i = len(tok(eq_i,  add_special_tokens=False)['input_ids'])
        n_j = len(tok(eq_j,  add_special_tokens=False)['input_ids'])
        n_c = len(tok(ctx,   add_special_tokens=False)['input_ids'])
        budget = self.max_length - 4 - n_i - n_j  # 4 special tokens
        if budget <= 0 or n_c == 0:
            return ''
        if n_c <= budget:
            return ctx
        ids  = tok(ctx, add_special_tokens=False)['input_ids']
        half = budget // 2
        keep = ids[:half] + ids[-(budget - half):]
        return tok.decode(keep, skip_special_tokens=True)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        p   = self.pairs[idx]
        tok = self.tokenizer
        sep = tok.sep_token or '[SEP]'
        ctx = self._fit_context(p['eq_i_text'], p['context'], p['eq_j_text'])
        text = f"{p['eq_i_text']} {sep} {ctx} {sep} {p['eq_j_text']}"
        enc  = tok(
            text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'scalar_features': torch.tensor(p['scalar_features'], dtype=torch.float),
            'label':          torch.tensor(p['label'],            dtype=torch.long),
        }


## 6. Model Definition

In [ ]:
class EncoderClassifier(nn.Module):
    """
    Encoder + classification head.
    [CLS] (768) || scalar (2) -> Linear(770,256) -> ReLU -> Dropout -> Linear(256,2)
    """

    def __init__(self, model_name: str, dropout: float = 0.1, n_scalar: int = 2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(h + n_scalar, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
        )

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        scalar:         torch.Tensor,
    ) -> torch.Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # SciBERT has pooler_output; DeBERTa-v3 may not
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            cls = out.pooler_output
        else:
            cls = out.last_hidden_state[:, 0, :]
        return self.head(torch.cat([cls, scalar], dim=-1))


## 7. Training Loop

In [ ]:
def pooled_metrics(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f


def run_epoch(model, loader, optimizer, scheduler, scaler, loss_fn, cfg, train=True):
    """One training or eval epoch. Returns (loss, precision, recall, f1)."""
    model.train(train)
    total_loss = tp = fp = fn = 0

    ctx_mgr = torch.enable_grad() if train else torch.no_grad()
    with ctx_mgr:
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            sclr  = batch['scalar_features'].to(DEVICE)
            lbls  = batch['label'].to(DEVICE)

            with autocast(enabled=torch.cuda.is_available()):
                logits = model(ids, mask, sclr)
                loss   = loss_fn(logits, lbls)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                if scheduler:
                    scheduler.step()

            total_loss += loss.item() * len(lbls)
            preds = logits.argmax(-1)
            tp += ((preds == 1) & (lbls == 1)).sum().item()
            fp += ((preds == 1) & (lbls == 0)).sum().item()
            fn += ((preds == 0) & (lbls == 1)).sum().item()

    avg_loss = total_loss / max(len(loader.dataset), 1)
    return avg_loss, *pooled_metrics(tp, fp, fn)


def predict_loader(model, loader, pairs, threshold=0.5):
    """Run inference on a DataLoader; return pairs with 'pred' and 'prob' added."""
    model.eval()
    probs_all = []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            sclr = batch['scalar_features'].to(DEVICE)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(ids, mask, sclr)
            probs_all.extend(torch.softmax(logits, -1)[:, 1].cpu().tolist())
    out = []
    for pair, prob in zip(pairs, probs_all):
        d = dict(pair)
        d['prob'] = prob
        d['pred'] = int(prob >= threshold)
        out.append(d)
    return out


def train_fold(train_pairs, val_pairs, tokenizer, cfg, fold_id, seed):
    """
    Train one CV fold. Returns (val_predictions_with_pred, best_model).
    Saves a checkpoint after every epoch so a disconnect can be resumed.
    """
    set_seed(seed)
    train_ds = PairDataset(train_pairs, tokenizer, cfg.MAX_LENGTH)
    val_ds   = PairDataset(val_pairs,   tokenizer, cfg.MAX_LENGTH)
    train_dl = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model   = EncoderClassifier(cfg.MODEL_NAME, cfg.DROPOUT).to(DEVICE)
    wt      = torch.tensor([1.0, cfg.POSITIVE_WEIGHT], device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=wt)

    enc_params  = list(model.encoder.parameters())
    head_params = list(model.head.parameters())
    optimizer   = torch.optim.AdamW([
        {'params': enc_params,  'lr': cfg.ENCODER_LR},
        {'params': head_params, 'lr': cfg.HEAD_LR},
    ])
    n_steps   = len(train_dl) * (1 if cfg.DEBUG_MODE else cfg.EPOCHS)
    warmup    = int(n_steps * cfg.WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, n_steps)
    scaler    = GradScaler(enabled=torch.cuda.is_available())

    best_f1, patience, best_state = -1.0, 0, None
    n_epochs = 1 if cfg.DEBUG_MODE else cfg.EPOCHS

    for ep in range(n_epochs):
        tr_loss, tr_p, tr_r, tr_f = run_epoch(model, train_dl, optimizer, scheduler, scaler, loss_fn, cfg, train=True)
        vl_loss, vl_p, vl_r, vl_f = run_epoch(model, val_dl,   optimizer, scheduler, scaler, loss_fn, cfg, train=False)
        logger.info(
            f'[{fold_id}] ep {ep+1}/{n_epochs} '
            f'tr_loss={tr_loss:.4f} tr_f1={tr_f:.4f} | '
            f'val_loss={vl_loss:.4f} val_p={vl_p:.4f} val_r={vl_r:.4f} val_f1={vl_f:.4f}'
        )
        torch.save(model.state_dict(), Path(cfg.CHECKPOINT_DIR) / f'{fold_id}_ep{ep+1}.pt')

        if vl_f > best_f1:
            best_f1    = vl_f
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
        else:
            patience += 1
            if patience >= cfg.PATIENCE and not cfg.DEBUG_MODE:
                logger.info(f'[{fold_id}] early stop at ep {ep+1}')
                break

    if best_state:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    return predict_loader(model, val_dl, val_pairs, cfg.THRESHOLD), model


## 8. Cross-Validation Orchestrator

In [ ]:
def run_cv(articles_used, all_pairs_per_article, tokenizer, cfg, seed):
    """5-fold article-level CV for one seed. Returns pooled val predictions."""
    kf       = KFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=42)
    n        = len(articles_used)
    all_preds = []
    n_folds  = 1 if cfg.DEBUG_MODE else cfg.N_FOLDS

    for fi, (tr_idx, vl_idx) in enumerate(kf.split(range(n))):
        if fi >= n_folds:
            break
        fold_id = f'seed{seed}_fold{fi+1}'
        tr_pairs = [p for i in tr_idx for p in all_pairs_per_article[i]]
        vl_pairs = [p for i in vl_idx for p in all_pairs_per_article[i]]
        logger.info(
            f'[{fold_id}] tr_arts={len(tr_idx)} vl_arts={len(vl_idx)} '
            f'tr_pairs={len(tr_pairs)} vl_pairs={len(vl_pairs)}'
        )
        val_preds, _ = train_fold(tr_pairs, vl_pairs, tokenizer, cfg, fold_id, seed)

        tp = sum(1 for p in val_preds if p['pred']==1 and p['label']==1)
        fp = sum(1 for p in val_preds if p['pred']==1 and p['label']==0)
        fn = sum(1 for p in val_preds if p['pred']==0 and p['label']==1)
        pr, rc, f1 = pooled_metrics(tp, fp, fn)
        logger.info(f'[{fold_id}] FOLD: P={pr:.4f} R={rc:.4f} F1={f1:.4f} TP={tp} FP={fp} FN={fn}')
        all_preds.extend(val_preds)

    return all_preds


def run_all_seeds(articles_used, all_pairs_per_article, cfg):
    """Run CV for every seed. Returns (per_seed_results, all_preds_by_seed)."""
    tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
    per_seed  = []
    by_seed   = {}
    seeds     = [cfg.SEEDS[0]] if cfg.DEBUG_MODE else cfg.SEEDS

    for seed in seeds:
        logger.info(f'===== SEED {seed} =====')
        set_seed(seed)
        preds = run_cv(articles_used, all_pairs_per_article, tokenizer, cfg, seed)
        tp = sum(1 for p in preds if p['pred']==1 and p['label']==1)
        fp = sum(1 for p in preds if p['pred']==1 and p['label']==0)
        fn = sum(1 for p in preds if p['pred']==0 and p['label']==1)
        pr, rc, f1 = pooled_metrics(tp, fp, fn)
        logger.info(f'SEED {seed} POOLED: P={pr:.4f} R={rc:.4f} F1={f1:.4f} TP={tp} FP={fp} FN={fn}')
        per_seed.append({'seed': seed, 'precision': round(pr,4), 'recall': round(rc,4),
                         'f1': round(f1,4), 'tp': tp, 'fp': fp, 'fn': fn})
        by_seed[seed] = preds

    return per_seed, by_seed


## 9. Evaluation

In [ ]:
def aggregate(per_seed):
    ps = [r['precision'] for r in per_seed]
    rs = [r['recall']    for r in per_seed]
    fs = [r['f1']        for r in per_seed]
    return {
        'precision_mean': round(float(np.mean(ps)), 4),
        'precision_std':  round(float(np.std(ps)),  4),
        'recall_mean':    round(float(np.mean(rs)), 4),
        'recall_std':     round(float(np.std(rs)),  4),
        'f1_mean':        round(float(np.mean(fs)), 4),
        'f1_std':         round(float(np.std(fs)),  4),
    }


def build_per_article_preds(by_seed, articles_used, article_data):
    """Per-article predicted and ground-truth edges (from last seed)."""
    last_preds = list(by_seed.values())[-1]
    by_art: Dict[str, list] = {aid: [] for aid in articles_used}
    for p in last_preds:
        by_art[p['article_id']].append(p)
    gt = {d['article_id']: d['adjacency_list'] for d in article_data}
    out = {}
    for aid in articles_used:
        pred_edges = [[p['vi'], p['vj']] for p in by_art.get(aid, []) if p['pred'] == 1]
        adj = gt[aid]
        gt_edges = [
            [src, tgt]
            for src, tgts in adj.items()
            for tgt in (tgts or [])
            if tgt is not None
        ]
        out[aid] = {'predicted_edges': pred_edges, 'ground_truth_edges': gt_edges}
    return out


## 10. Results Saving

In [ ]:
def save_results(cfg, per_seed, by_seed, articles_used, article_data):
    agg        = aggregate(per_seed)
    per_art    = build_per_article_preds(by_seed, articles_used, article_data)
    output = {
        'model_name':              cfg.MODEL_NAME,
        'config': {
            'max_length':      cfg.MAX_LENGTH,
            'batch_size':      cfg.BATCH_SIZE,
            'epochs':          cfg.EPOCHS,
            'encoder_lr':      cfg.ENCODER_LR,
            'head_lr':         cfg.HEAD_LR,
            'positive_weight': cfg.POSITIVE_WEIGHT,
            'n_folds':         cfg.N_FOLDS,
            'seeds':           cfg.SEEDS,
            'patience':        cfg.PATIENCE,
            'debug_mode':      cfg.DEBUG_MODE,
        },
        'per_seed_results':        per_seed,
        'aggregated':              agg,
        'per_article_predictions': per_art,
    }
    with open(cfg.RESULTS_PATH, 'w') as f:
        json.dump(output, f, indent=2)
    logger.info(f'Saved -> {cfg.RESULTS_PATH}')

    print('\n===== FINAL RESULTS =====')
    print(f'Model: {cfg.MODEL_NAME}  |  Articles: {len(articles_used)}')
    header = f'{"Seed":>6}  {"Prec":>8}  {"Rec":>8}  {"F1":>8}  {"TP":>6}  {"FP":>6}  {"FN":>6}'
    print(header)
    for r in per_seed:
        print(f'{r["seed"]:>6}  {r["precision"]:>8.4f}  {r["recall"]:>8.4f}  {r["f1"]:>8.4f}  {r["tp"]:>6}  {r["fp"]:>6}  {r["fn"]:>6}')
    print(f'{"mean":>6}  {agg["precision_mean"]:>8.4f}  {agg["recall_mean"]:>8.4f}  {agg["f1_mean"]:>8.4f}')
    print(f'{"±std":>6}  {agg["precision_std"]:>8.4f}  {agg["recall_std"]:>8.4f}  {agg["f1_std"]:>8.4f}')


## 11. Sanity Checks

In [ ]:
def print_sanity_checks(by_seed, n=10):
    """Print sample TP, FP, and FN predictions for manual inspection."""
    preds = list(by_seed.values())[-1]
    tps = [p for p in preds if p['pred']==1 and p['label']==1][:n]
    fps = [p for p in preds if p['pred']==1 and p['label']==0][:n]
    fns = [p for p in preds if p['pred']==0 and p['label']==1][:n]

    for tag, subset in [('TP', tps), ('FP', fps), ('FN', fns)]:
        print(f'\n-- {tag} --')
        for p in subset:
            print(f'  {p["article_id"]}  {p["vi"]} -> {p["vj"]}  prob={p["prob"]:.3f}')
            print(f'    eq_i: {p["eq_i_text"][:90]}')
            print(f'    eq_j: {p["eq_j_text"][:90]}')


## 12. Run

In [ ]:
def main():
    """
    Full pipeline. Set cfg.DEBUG_MODE=False before the real run.
    Checkpoints are saved after every epoch so a Colab disconnect can be resumed.
    """
    t0 = time.time()
    if cfg.DEBUG_MODE:
        logger.info('DEBUG_MODE=True: 1 fold, 1 epoch, seed 42 only')

    per_seed, by_seed = run_all_seeds(articles_used, all_pairs_per_article, cfg)
    save_results(cfg, per_seed, by_seed, articles_used, article_data)
    print_sanity_checks(by_seed)

    logger.info(f'Done in {(time.time()-t0)/60:.1f} min')


main()
